# Stage 2 — SAW Recommender

**What this notebook does:** combines the two signals — *weak now* (Stage 0) and *will worsen* (Stage 1) — and uses **Simple Additive Weighting (SAW)** to rank the 5 cognitive domains for training, with transparent reasons. A separate **adapter** (built later in this notebook) routes each patient to the **AD arm**, **LBD arm**, or a **safe fallback**, based on the diagnosis probabilities from Components 1–3.

**Plan:** 1) build the criteria table → 2) SAW scoring (normalise → weight → rank → reasons) → 3) adapter + mock + safe fallback → 4) end-to-end → 5) evaluation (with/without decline, vs baseline).

**This step (Step 1):** assemble the per-patient **criteria table** — the inputs SAW will score. No scoring yet.

## Setup

Load the two signals saved by the earlier stages, and set the weakness threshold.

The **weakness threshold is a cited clinical convention** a domain is "weak" when `z ≤ −1.0 SD`. We never tune it to improve results; we will instead run a sensitivity analysis (−1.0 / −1.5 / −2.0) later.

In [ ]:
import os, json
import numpy as np, pandas as pd

PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

# CITED clinical convention (CLAUDE_threshold_v1.md) -- DO NOT tune/auto-optimise.
# NIA-AA 1.0-1.5 SD; Heaton 1 SD; MoCA-Tamil < -1 SD.  z <= -1.0 flags a weak domain.
WEAK_THRESHOLD = -1.0

prof = pd.read_parquet(os.path.join(PROJ, "data", "cognitive_profiles.parquet"))      # weak-now: domain z-scores
decl = pd.read_parquet(os.path.join(PROJ, "data", "decline_predictions_oof.parquet"))  # will-worsen: OOF severities (leak-free)
print("profiles:", prof.shape, "| decline (out-of-fold):", decl.shape)

## Step 1 — Build the SAW criteria table

For each patient and each of the 5 domains we assemble the three inputs SAW will use:

- **weak-now severity** = `−z` — higher means more impaired *right now* (from Stage 0).
- **is_weak** = `z ≤ −1.0` — the cited weakness flag. We use it as a **criterion** (option B), so a domain predicted to *worsen into* weakness can still surface.
- **will-worsen severity** = the Stage 1 prediction — higher means faster predicted decline. We use the **out-of-fold** values so any later evaluation is leakage-free.

We keep every profiled patient. The will-worsen columns are **blank for patients without enough longitudinal data** (one visit, or no two visits that overlap on a domain) — SAW handles that in the next step. The two severities are on **different scales** (weak-now can be large; will-worsen ≈ 0–1.7), so we leave them **raw** here; SAW normalises them **within each patient** at the scoring step.

**What the weak-% table means:** it is **descriptive** — how common the flag is in each arm — **not** proof of clinical optimality or of "disease signatures" (those claims need separate clinical validation). Also note for the methodology: z-scores use the **NACC cognitively-normal participants in this dataset, without age/education adjustment**.

In [ ]:
C = prof[["NACCID", "diagnosis"]].copy()
for d in dom:
    C["weaknow_" + d] = -prof[d]                      # higher = weaker now
    C["isweak_" + d]  = prof[d] <= WEAK_THRESHOLD     # cited -1.0 SD flag (as a criterion)

C = C.merge(decl[["NACCID"] + ["severity_" + d for d in dom]], on="NACCID", how="left")
C = C.rename(columns={"severity_" + d: "willworsen_" + d for d in dom})

OUT = os.path.join(PROJ, "data", "saw_criteria.parquet")
C.to_parquet(OUT, index=False)
print("Saved:", OUT)

has_decline = C[["willworsen_" + d for d in dom]].notna().any(axis=1)
print(f"\nPatients: {len(C):,} total | with a will-worsen signal: {has_decline.sum():,}")

print("\n% of patients with each domain flagged WEAK (z <= -1.0), by arm:")
flag = C.groupby("diagnosis")[["isweak_" + d for d in dom]].mean().mul(100).round(0)
flag.columns = dom
print(flag.to_string())

print("\nPreview (memory & visuospatial columns):")
cols = ["NACCID", "diagnosis", "weaknow_memory", "isweak_memory", "willworsen_memory",
        "weaknow_visuospatial", "isweak_visuospatial", "willworsen_visuospatial"]
print(C.head(4)[cols].round(2).to_string(index=False))

## Step 2 — SAW scoring rules (defined before coding)

We rank the 5 domains per patient by a weighted sum of two criteria. Rules fixed in writing so they're defensible:

**Criteria (continuous, per domain):**
- **C1 weak-now** = `−z` (higher = more impaired now).
- **C2 will-worsen** = out-of-fold decline severity (higher = faster predicted decline).
- `is_weak` (z ≤ −1.0, cited) is used ONLY in the reason text and the sensitivity analysis — **not** as a third weighted criterion (avoids double-counting weakness).

**Normalisation:** within each patient, **min–max** each criterion across its 5 domains → 0–1. If all 5 are equal (or, for C2, fewer than 2 domains have a value), that criterion contributes 0.
*What the score means:* a **relative training priority within the patient, not absolute clinical severity** — the weakest domain scores 1 even if all domains are normal. So we also report the raw `is_weak` flags and decline values.

**Weights:** default **0.5 / 0.5** (weight-sensitivity analysis later).

**Missing decline:**
- *No decline for any domain* → **weak-now-only** ranking (output labelled).
- *Decline missing for some domains* → those domains scored on weak-now only, **weights renormalised per domain**. *Limitation:* domains scored on different criteria sit on the same 0–1 scale but are **not fully equivalent in meaning** — we record the criteria used per domain.

**Maintenance:** if a patient has **no weak domain and nothing predicted to worsen** → label the output `maintenance` (no clinical target).

**Logging:** per patient record the pathway (both / weak-now-only) and report the frequencies.

In [ ]:
import warnings
crit = pd.read_parquet(os.path.join(PROJ, "data", "saw_criteria.parquet"))

WN = crit[["weaknow_" + d for d in dom]].to_numpy(float)     # weak-now severity (-z)
WW = crit[["willworsen_" + d for d in dom]].to_numpy(float)  # will-worsen severity (NaN if no decline)
IW = crit[["isweak_" + d for d in dom]].to_numpy(bool)
w_now, w_dec = 0.5, 0.5

# C1: within-patient min-max (weak-now always present)
lo, hi = WN.min(1, keepdims=True), WN.max(1, keepdims=True); rng = hi - lo
nNOW = np.where(rng > 0, (WN - lo) / np.where(rng > 0, rng, 1), 0.0)

# C2: within-patient min-max over available decline values (need >=2 to normalise)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    lo2, hi2 = np.nanmin(WW, 1, keepdims=True), np.nanmax(WW, 1, keepdims=True)
navail, rng2 = (~np.isnan(WW)).sum(1, keepdims=True), hi2 - lo2
usable = (navail >= 2) & (rng2 > 0)
nDEC = np.where(usable, (WW - lo2) / np.where(rng2 > 0, rng2, 1), np.nan)

# per-domain score with per-domain renormalised weights
has_dec = ~np.isnan(nDEC)
num = w_now * nNOW + np.where(has_dec, w_dec * np.nan_to_num(nDEC), 0.0)
den = w_now + np.where(has_dec, w_dec, 0.0)
SCORE = num / den

pathway = np.where(has_dec.any(1), "both", "weak-now-only")
maintenance = (~IW.any(1)) & (np.nan_to_num(WW) <= 0).all(1)

rec = crit[["NACCID", "diagnosis"]].copy()
for j, d in enumerate(dom):
    rec["score_" + d] = SCORE[:, j]
rec["top_domain"]  = [dom[i] for i in SCORE.argmax(1)]
rec["pathway"]     = pathway
rec["maintenance"] = maintenance
rec.to_parquet(os.path.join(PROJ, "data", "saw_recommendations.parquet"), index=False)
print("Saved: data/saw_recommendations.parquet")

print("\nPathway used:", dict(pd.Series(pathway).value_counts()))
print("Maintenance-labelled:", int(maintenance.sum()))
print("\nTop-priority domain by arm (count):")
print(rec.groupby("diagnosis")["top_domain"].value_counts().unstack().fillna(0).astype(int).to_string())

def reason(iw, ww):
    worsening = (not np.isnan(ww)) and ww > 0
    if iw and worsening: return "weak now + predicted decline -> high priority"
    if iw:               return "weak now -> priority"
    if worsening:        return "not weak now but predicted decline -> preventive"
    return "stable"

ex = int(np.where(pathway == "both")[0][0])
print(f"\n--- Example plan ({crit.iloc[ex]['NACCID']}, {crit.iloc[ex]['diagnosis']}, pathway={pathway[ex]}) ---")
for i in SCORE[ex].argsort()[::-1]:
    print(f"  {dom[i]:<13} score={SCORE[ex,i]:.2f}  | {reason(IW[ex,i], WW[ex,i])}")